# Практические задания для первого блока по базовому курсу по Python

## Задание 1: Анализ данных из текстового файла

In [1]:
import logging
import pandas as pd
from openpyxl import load_workbook
import os


# Создание дерикторий если их нет и настройка логирования
try:
    os.makedirs('./log', exist_ok=True)
    os.makedirs('./data', exist_ok=True)
    os.makedirs('./output_data', exist_ok=True)
    
    logging.basicConfig(
        filename='./log/app_task1.log',
        filemode='w',
        level=logging.DEBUG,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True,
        encoding='UTF-8',
    )
except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')


#Константы
PATH_TXT_FILE = './data/data_task1.txt'
PATH_OUTPUT_FILE = './output_data/table_output_task1.xlsx'


def count_product(path_txt_file):
    """ Функция обработки данных из файла.

    Читает данные их файла data.txt, преобразует данные из списка в словарь, проводит расчеты дохода по товарам и общего дохода.
    Возвращает словарь и общий доход.
    """

    with open(path_txt_file, 'r', encoding='UTF-8') as file:
        content = file.readlines()
        logging.info('Чтение файла прошло успешно')
        products_data = {}

        for row in content:
            product, count, price = row.strip().split(',')
            products_data[product] ={
                'count': int(count),
                'price': float(price),
            }
        logging.info(f'Обработано товаров: {len(products_data)}')
        total_income = 0

        for data in products_data.values():
            total_revenue_product = data['count'] * data['price']
            data['total_revenue_product'] = round(total_revenue_product, 2)
            total_income += data['total_revenue_product']
            total_income = round(total_income, 2)

        logging.info(f'Расчет дохода завершен. Он составляет : {total_income}')

        return products_data, total_income


def create_excel_table(path_output_file, products_data, total_income):
    """ Функция создания excel таблицы с результатами расчета по данным файла data.txt .

    Принимает словарь с данными и общий доход, формирует DataFrame из списка, записывает DataFrame в excel файл.
    Возвращает excel таблицу с комментарием после последний сроки о результатах общих продаж.
    """

    logging.info('Создание Excel таблицы')

    # Создаем DataFrame
    df = pd.DataFrame.from_dict(
        products_data,
        orient='index'
    )

    df.index.name = 'Товар'

    df = df.rename(columns={
    'count': 'Количество',
    'price': 'Цена',
    'total_revenue_product': 'Выручка'
    }
    )

    #  Создаем и заполняем Excel файл
    with pd.ExcelWriter(path_output_file) as writer:
        df.to_excel(writer, index=True)

    wb = load_workbook(path_output_file)
    sheet = wb.active


    if total_income > 1000000:
        message = f'Общий доход равен : {total_income}, хорошо заработали))'
    else:
        message = f'Общий доход равен : {total_income}, в следующий раз надо постараться'

    message_row = sheet.max_row + 1

    sheet.cell(
        row=message_row,  # Строка для сообщения
        column=1,  # Колонка
        value=message,  # Сообщение
    )

    wb.save(path_output_file)
    logging.info('Excel файл сохранен успешно')

    # Читаем excel файл и возвращаем результат
    with pd.ExcelFile(path_output_file) as xls:
        df = pd.read_excel(xls, 'Sheet1')

    return (df)


try:
    
    products_data, total_income = count_product(PATH_TXT_FILE)
    print(create_excel_table(PATH_OUTPUT_FILE, products_data, total_income))

except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')
except TypeError as e:
    logging.critical(f'Ошибка обработки данных: {e}')
except ValueError as e:
    logging.error(f'Ошибка формата данных: {e}')



                                               Товар  Количество      Цена  \
0                                          Телевизор         5.0  29999.99   
1                                        Холодильник         2.0  49999.50   
2                                           Смартфон        10.0  19999.00   
3                                            Ноутбук         3.0  69999.99   
4                                            Пылесос         7.0   8999.90   
5                                  Стиральная машина         4.0  35999.95   
6                                      Микроволновка         8.0   7999.99   
7                                          Кофеварка        15.0   4999.50   
8  Общий доход равен : 1005980.44, хорошо заработ...         NaN       NaN   

     Выручка  
0  149999.95  
1   99999.00  
2  199990.00  
3  209999.97  
4   62999.30  
5  143999.80  
6   63999.92  
7   74992.50  
8        NaN  


## Задание 2: Обработка данных из текстового файла

In [2]:
# Импорты
import logging
import os


# Настройка логирования и создания дерикторий если их нет
try:
    os.makedirs('./log', exist_ok=True)
    os.makedirs('./data', exist_ok=True)
    os.makedirs('./output_data', exist_ok=True)
    
    logging.basicConfig(
        filename='./log/task2.log',
        filemode='w',
        level=logging.DEBUG,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True,
        encoding='UTF-8',
    )
except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')


# Константы
PATH_TXT_FILE = './data/students.txt'
PATH_OUTPUT_FILE = './output_data/top_students.txt'


def average_score(students) -> dict:
    """Функция принимает на вход файл со списком студентов и их оценками
    Обрабатывает данные из файла, расчитывает средний балл каждого студента
    Возовращает словарь с данными о студентах
    """
    
    with open(students, 'r', encoding='UTF-8') as file:
        content = file.readlines()
        logging.info('Чтение файла завершено успешно')
        students_data = {}

        logging.info('Обработка данных...')
        for row in content:
            student, *scores = row.strip().split(',')
            students_data[student] = {
                'scores': [int(grade) for grade in scores],
            }

            #  Расчет среднего балла каждого студента.
            students_data[student]['average score'] = sum(students_data[student]['scores']) / len(students_data[student]['scores'])
        
        logging.info(f'Данные в файле обоработаны успешно, посчитано {len(students_data.keys())} студентов')
        return students_data


def top_students(path_output_file: str ,score_students: dict):
    """Функция принимает на вход словарь с данными о студентах
    Сортирует данные в файле по среднему баллу от большего к меньшему
    Записывает в новый файл студентов и их балл у которых средний балл выше threshold_top
    """

    threshold_top = 4.5
    
    logging.info('Сортировка студентов по среднему баллу...')
    score_students = sorted(score_students.items(),
                            key=lambda data: data[1]['average score'],  #  Передан ключ который сообщай о том , что нужно сортировать по среднему баллу.
                            reverse=True  #  Перевернули.
                           )
    logging.info('Сортировка прошла успешно')
    
    with open(path_output_file, 'w', encoding='UTF-8') as top_file:
        logging.info('Начинает запись данных в файл top_students.txt')
        
        top_file.write(f'Топ студентов: \n')
        count_top_students = 0

        for student, data in score_students:
            if data['average score'] > threshold_top:
                count_top_students += 1
                top_file.write(f'{count_top_students}. {student} - средний балл: {data["average score"]}\n')
                

        logging.info(f'Запись лучших студентов с баллом выше {threshold_top} завершена. В топ лучших вошло {count_top_students} студента')


try:
    top_students(PATH_OUTPUT_FILE, average_score(PATH_TXT_FILE))
    
    with open(PATH_OUTPUT_FILE, 'r', encoding='UTF-8') as top_students:
        top = top_students.read()
        print(top)

except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')
except TypeError as e:
    logging.critical(f'Ошибка обработки данных: {e}')
except ValueError as e:
    logging.error(f'Ошибка формата данных: {e}')

Топ студентов: 
1. Анна Соколова - средний балл: 5.0
2. Елена Кузнецова - средний балл: 4.6
3. Мария Козлова - средний балл: 4.6
4. Сергей Новиков - средний балл: 4.6



## Задание 3: Создание простой базы данных в текстовом файле

In [3]:
# Импорты
import logging
import os


# Создание директорий и настрйока логирования
try:
    os.makedirs('./log', exist_ok=True)
    os.makedirs('./data', exist_ok=True)
    
    logging.basicConfig(
        filename='./log/to_do_list.log',
        filemode='w',
        level=logging.DEBUG,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True,
        encoding='UTF-8',
    )
except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')


# Реализация класса To_Do_list
class To_Do_list():
    """Класс To_Do_list - предсталяет сабой реализацию todo листа со списками задач и функционалом
    Функционал: Добавление задачи, удаление, удаление всех задач и добавление новой, замена задачи
    """
    
    def __init__(self):
        """ Иниациализатор атрибутов обьекта класса """
        
        self.__to_do_list = []
        self.__path_file = './data/tasks.txt'
        logging.info('Список задач успешно создан')
        
        with open(self.__path_file, 'r', encoding='UTF-8') as tasks:
            file_tasks = tasks.readlines()
            
            for task in file_tasks:
                task = task.strip()
                if ': ' in task:
                    task = task.split(': ', 1)[1]
                    self.__to_do_list.append(task)
                elif task:
                    self.__to_do_list.append(task)
                    
            logging.info('Имеющиеся задачи успешно сохранены в список to_do_list')
                    
    def save_tasks(self):
        """Функция сохраняет список задач из to_do_list в tasks.txt в виде нумерованого списка"""
        
        with open(self.__path_file, 'w', encoding='UTF-8') as tasks:
            for number, task in enumerate(self.__to_do_list, start=1):
                tasks.write(f'{number}: {task}\n')
            logging.info('Задачи из to_do_list успешно сохранены записаны в файл tasks в виде нумерованого списка')
    
    @property
    def get_task(self):
        """Функция возвращает список задач из tasks.txt"""
        
        with open(self.__path_file , 'r', encoding='UTF-8') as tasks:
            file_tasks = tasks.read()
            logging.info('Список задач успешно прочитан и выведен на экран')
            return file_tasks

    def get_task_by_id(self, id_task: int) -> str:
        """Функция возвращает задачу по ее ID"""
        
        if 1 <= id_task <= len(self.__to_do_list):
            return self.__to_do_list[id_task - 1]
        else:
            raise IndexError('Задача c данным id отсутсвует')

    
    def post_task(self, new_task: str):
        """Функция добавляет новую задачу в to_do_list и в файл tasks.txt"""
        
        if isinstance(new_task, str):
            self.__to_do_list.append(new_task)
            logging.info('Задача успешно сохранена в список to_do_list')
            
            with open(self.__path_file, 'a', encoding='UTF-8') as tasks:
                number_task = len(self.__to_do_list)
                tasks.write(f'{number_task}: {new_task}\n')
                logging.info('Текст задачи успешно принят, записан в tasks')
        else:
            raise TypeError('Отсутсвует либо некорректный тип данных задачи')

    
    def put_task(self, new_task: str):
        """Функция полностью чистит список to_do_list и tasks.txt и добавляет новую задачу"""
        
        if isinstance(new_task, str):
            self.__to_do_list.clear()
            self.__to_do_list.append(new_task)
            self.save_tasks()
            logging.info('Список задач успешно перезаписан')
        else:
            raise TypeError('Отсутсвует либо некорректный тип данных задачи')

    
    def patch_task(self, id_task:int, new_task: str):
        """Функция заменяет указаную задачу на  новую"""
        
        if 1 <= id_task <= len(self.__to_do_list):
            old_task = self.__to_do_list[id_task - 1]
            self.__to_do_list[id_task - 1] = new_task
            logging.info(f'Задача {old_task} успешно заменена на {new_task}')
        else:
            raise ValueError('Задачи с таким номером нет')
        
        self.save_tasks()
    
    def delete_task(self, id_task=None, task=None):
        """Функция удаляет задачу из to_do_list и перезаписывает tasks.txt в нужном порядке нумерации"""
        
        if 1 <= id_task <= len(self.__to_do_list):
            delete_task = self.__to_do_list.pop(id_task - 1)
            self.save_tasks()
            logging.info(f'Задача {delete_task} успешно удалена, список задач сохранен')
        else:
            raise TypeError('Задача для удаления отсутвует или неверного типа данных')


try:
    todo = To_Do_list()
    todo.save_tasks()
    logging.info('Обьект класса To_Do_list успешно создан')

    # Пользовательский интерфейс
    while True:
        user_input = int(input(
            'Введите цифру задачи операции которую хотите выполнить?\n'
            '1) Добавить задачу\n'
            '2) Удалить все задачи и добавить новую\n'
            '3) Удалить задачу\n'
            '4) Заменить задачу\n'
            '5) Вывести весь список задач\n'
            '6) Нажмите любую кнопку чтобы выйти\n'
        )
        )
        
        if user_input == 1:
            logging.info('Пользователь выбрал пункт - Добавить задачу')
            add_input =input('Введите текст задачи: ')
            
            logging.info(f'Пользователь добавляет задачу {add_input}')
            
            todo.post_task(add_input)
            print(todo.get_task)

        elif user_input == 2:
            logging.info('Пользователь выбрал пункт - Удалить все задачи и добавить новую')
            add_delall_input =input('Введите текст новой задачи (все остальные будут удалены): ')
            
            logging.info(f'Пользователь удалил все задачи и добавляет задачу {add_delall_input}')
            
            todo.put_task(add_delall_input)
            print(todo.get_task)

        elif user_input == 3:
            print(todo.get_task)
            logging.info(f'Пользователь выбрал пункт - Удалить задачу')
            
            del_input = int(input('Введите номер задачи которую хотите удалить: '))
            del_task = todo.get_task_by_id(del_input)
            
            logging.info(f'Пользователь удаляет задачу {del_task}')
            
            todo.delete_task(del_input)
            print(todo.get_task)
            
        elif user_input == 4:
            logging.info(f'Пользователь выбрал пункт - Заменить задачу')
            
            patch_input = int(input('Введите номер задачи которую хотите заменить: '))
            new_task = input('Введите текст новой задачи: ')
            patch_task = todo.get_task_by_id(patch_input)
            
            logging.info(f'Пользователь меняет задачу {patch_task} на {new_task}')
        
            todo.patch_task(patch_input, new_task)
            print(todo.get_task)

        elif user_input == 5:
            print(todo.get_task)

        else:
            logging.info(f'Пользователь выходит')
            break

except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')
except TypeError as e:
    logging.critical(f'Ошибка обработки данных: {e}')
except ValueError as e:
    logging.error(f'Ошибка формата данных: {e}')
except IndexError as e:
    logging.critical(f'Ошибка обработки данных: {e}')
        

Введите цифру задачи операции которую хотите выполнить?
1) Добавить задачу
2) Удалить все задачи и добавить новую
3) Удалить задачу
4) Заменить задачу
5) Вывести весь список задач
6) Нажмите любую кнопку чтобы выйти
 6


## Задание 4: Обработка данных о сотрудниках


In [67]:
# Импорты
import logging
import os


# Настройка логирования и создание директорий
try:
    os.makedirs('./log', exist_ok=True)
    os.makedirs('./data', exist_ok=True)
    os.makedirs('./output_data', exist_ok=True)
    
    logging.basicConfig(
        filename='./log/task3.log',
        filemode='w',
        level=logging.DEBUG,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True,
        encoding='UTF-8',
    )
except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')

#Констаны
PATH_TXT_FILE = './data/employees.txt'
PATH_OUTPUT_FILE = './output_data/high_earners.txt'


def filter_employees(data):
    """ Фукция принимает файл с данным, обрабатывает в словарь, расчитывает среднюю зарплату сотрудников
    сравнивает запрплату каждого сотрудника со средней, сортирует по запрлате и соотношению Зарплата сотрудника - Средняя зарплата
    Взвращает список содержащий только тех, у кого зарплата больше средней
    """
    with open(data, 'r', encoding='UTF-8') as input_file:
        logging.info('Файл успешно прочитан. Обработка данных ...')
        
        employees = {}
        
        for line in input_file:
            employer, age, position, salary = line.strip().split(',')
            employees[employer] ={
                'age': age,
                'position': position,
                'salary': int(salary),
            }
        
    сomparison_average_salary = sum([employer['salary'] for employer in employees.values()]) / len(employees.keys())

    for employer, data in employees.items():
       data['сomparison_average_salary'] = data['salary'] > сomparison_average_salary

    logging.info(f'Обработка данных заверешена')
    logging.info(
        f'Обработано {len(employees.keys())} сотрудников. '   
        f'Cредняя зарплата: {сomparison_average_salary}. '
        f'У {len([employer['сomparison_average_salary'] for employer in employees.values() if employer['сomparison_average_salary']== True])} сотрудников зарплата больше средней. '
    )

    # Сортируем и получаем сортированый список состоящий из кортежей в которых 1 элелемент - Сотрудник, 2 - словарь с его данными. Так будет намного проще работать с данными при записи в файл
    sort_employees = sorted(employees.items(),
                    key=lambda data: (
                        data[1]['сomparison_average_salary'],
                        data[1]['salary']
                    ),
                    reverse=True  #  Перевернули.
            )

    logging.info(f'Словарь отсортирован ')

    # Фильтруем список и оставляем только тех сотрудников у который зарплата выше средней
    filtered_employees = [
        (employer, data)
        for employer, data in sort_employees
        if data['сomparison_average_salary']== True
    ]
    
    logging.info(f'Словарь отфильтрован по зарплате и соотношению Зарплата - Средняя зарплата: {filtered_employees}')

    return filtered_employees
    

def writer_in_output_file(PATH_OUTPUT_FILE, sort_employees):
    """Функция записи в файл данных о сотрбудника у которых зарплата выше средней"""
    
    logging.info(f'Данные успешно передены на запись в {PATH_OUTPUT_FILE}')
    
    with open(PATH_OUTPUT_FILE , 'w', encoding='UTF-8') as output_file:
        output_file.write('Сотрудники у которых зарплата выше средней:\n')
        for employer, data in sort_employees:
            if data['сomparison_average_salary']:
                output_file.write(f'{employer} - {data["position"]} - {data["salary"]}\n')

        logging.info(f'Данные успешно записаны')


try:

    high_earners = writer_in_output_file(PATH_OUTPUT_FILE, filter_employees(PATH_TXT_FILE))
    with open(PATH_OUTPUT_FILE, 'r', encoding='UTF-8') as output_file:
        print(output_file.read())
        logging.info(f'Данные успешно вывены на экран')

except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')
except TypeError as e:
    logging.critical(f'Ошибка обработки данных: {e}')
except ValueError as e:
    logging.error(f'Ошибка формата данных: {e}')



Сотрудники у которых зарплата выше средней:
Игорь Лебедев - Директор - 120000
Алексей Смирнов - Менеджер - 90000
Елена Кузнецова - Разработчик - 85000
Мария Петрова - Аналитик - 80000



## Задание 5: Словарь синонимов

In [69]:
# Импорты
import logging
import os


# Настройка логирования и создание директорий
try:
    os.makedirs('./log', exist_ok=True)
    os.makedirs('./data', exist_ok=True)
    os.makedirs('./output_data', exist_ok=True)
    
    logging.basicConfig(
        filename='./log/task5.log',
        filemode='w',
        level=logging.DEBUG,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True,
        encoding='UTF-8',
    )
except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')

#Констаны
PATH_TXT_FILE = './data/synonyms.txt'

class NotFoundWord(Exception):
    """ Пользовательская ошибка """
    pass


def get_synonyms(PATH_TXT_FILE, word):
    """ Функция поиска синонимов по слову
    Принимает слово, обрабатывает данные из файла ./data/synonyms.txt в словарь, производит поиск
    Возвращает синонимы нужного слова
    """

    
    with open(PATH_TXT_FILE, 'r', encoding='UTF-8') as db_synonyms:
        logging.info(f'Начинается поиск синонимов слова: {word}')
        synonyms = {}

        logging.info(f'Обработка данных ...')
        for line in db_synonyms:
            word_db, data = line.strip().split('-')
            synonyms[word_db] ={
                'synonym': [synonym for synonym in data.split(',')]
            }
        logging.info(f'Данные обработаны, нажинается поиск нужных синонимов...')
            
        for word_db, data_db in synonyms.items():
            
            if word_db.strip().lower() == word.strip().lower():
                result = []
                
                for indx, synonym in enumerate(data_db['synonym'], start=1):
                    result.append(f'{indx}. {synonym.strip()}')
                    
                logging.info(f'Синонимы слова "{word}" найдены : {result}')
                return ', '.join(result)
    raise NotFoundWord('Извините, наш словарь еще не такой большой в нем нет данного слова, либо вы ввели его не правильно')
    



while True:
    user_input = input('Введите слово, синоним которого хотите узнать:')
    print('Введите "q" если хотите выйти')
    logging.info(f'Пользователь запросил синонимы слова: {user_input}')

    if user_input.lower()=='q':
        print('Программа завершена')
        logging.info(f'Пользователь выходит')
        break
        
    try:
        synonyms = get_synonyms(PATH_TXT_FILE,user_input)
        print(synonyms)
        logging.info(f'Синонимы успешно выведены на экран: {synonyms}')
        
    except FileNotFoundError as e:
        print('Ошибка: файл со словарём не найден.')
        logging.critical(f'Ошибка, файл не найден: {e}')
        break
    except NotFoundWord as e:
        print(e)
        logging.warning(f'Ошибка обработки данных: {e}')
    except TypeError as e:
        logging.critical(f'Ошибка обработки данных: {e}')
        break
    except ValueError as e:
        logging.error(f'Ошибка формата данных: {e}')
        break


Введите слово, синоним которого хотите узнать: маленький


Введите "q" если хотите выйти
1. крошечный, 2. небольшой, 3. миниатюрный, 4. малогабаритный, 5. компактный


Введите слово, синоним которого хотите узнать: Большой


Введите "q" если хотите выйти
1. огромный, 2. крупный, 3. громадный, 4. масштабный, 5. величавый


Введите слово, синоним которого хотите узнать: й


Введите "q" если хотите выйти
Извините, наш словарь еще не такой большой в нем нет данного слова, либо вы ввели его не правильно


Введите слово, синоним которого хотите узнать: q


Введите "q" если хотите выйти
Программа завершена
